# Practice Lab: Multimodal Models (Lesson 6.5)

Module 6 · 8 exercises · VLMs + Gemini Vision
**Needs:** google-generativeai, pydantic, Pillow, transformers

Exercises 2, 3, 8 run locally (pure Python).


# Lesson 6.5: Multimodal Models — Gemini Vision & VLMs

8 exercises: 2E + 3M + 3C
**Needs:** `pip install google-generativeai pydantic Pillow transformers`

Exercises 2 (architecture), 3 (Pydantic schemas), and 8 (crop DB) run **locally**.


In [ ]:
!pip install google-generativeai pydantic Pillow transformers -q


In [ ]:
import os, json
from typing import Optional


---
## Exercise 2 (Easy): VLM Architecture Diagram

Map 5 VLM architectures. Pure Python data structure.


In [ ]:
# YOUR CODE
# TODO: create architectures dict
# TODO: calculate param distribution
# TODO: print formatted table


<details><summary>Solution</summary>


In [ ]:
architectures = {
    'LLaVA-1.5': {
        'vision': 'CLIP ViT-L/14', 'vision_params': 304,
        'projector': '2-layer MLP', 'proj_params': 20,
        'llm': 'Vicuna-13B', 'llm_params': 13000},
    'LLaVA-OneVision': {
        'vision': 'SigLIP-SO400M', 'vision_params': 400,
        'projector': 'MLP', 'proj_params': 20,
        'llm': 'Qwen2-7B', 'llm_params': 7000},
    'InternVL-3.5': {
        'vision': 'InternViT-6B', 'vision_params': 6000,
        'projector': 'QLLaMA', 'proj_params': 200,
        'llm': 'InternLM2.5-78B', 'llm_params': 78000},
    'Qwen2.5-VL': {
        'vision': 'Custom ViT 600M', 'vision_params': 600,
        'projector': 'Native', 'proj_params': 0,
        'llm': 'Qwen2.5-7B', 'llm_params': 7000},
    'PaliGemma-2': {
        'vision': 'SigLIP-SO400M', 'vision_params': 400,
        'projector': 'Linear', 'proj_params': 3,
        'llm': 'Gemma-2-3B', 'llm_params': 3000},
}

print(f"{'Model':<20} {'Vision':<18} "
      f"{'Projector':<14} {'LLM':<18} "
      f"{'Total(M)':<10} {'Vision%'}")
print('=' * 100)
for name, a in architectures.items():
    total = (a['vision_params'] + a['proj_params']
             + a['llm_params'])
    vpct = a['vision_params'] / total * 100
    ppct = a['proj_params'] / total * 100
    lpct = a['llm_params'] / total * 100
    print(f"{name:<20} {a['vision']:<18} "
          f"{a['projector']:<14} {a['llm']:<18} "
          f"{total:<10,} {vpct:.1f}%")

print()
print('Key insights:')
print('  - LLM dominates params (88-98%)')
print('  - Vision encoder is 2-12% of total')
print('  - Projector is <1% (MLP) to 0.2% (QLLaMA)')
print('  - InternVL has largest vision encoder (6B)')
print('  - PaliGemma has highest vision% (small LLM)')


</details>


---
## Exercise 3 (Medium): Document AI for Indian IDs

Pydantic schemas + validation. Pure Python.


In [ ]:
# YOUR CODE
# TODO: define AadhaarCard, PANCard, UtilityBill
# TODO: validate sample data


<details><summary>Solution (schema + validation, no API)</summary>


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class AadhaarCard(BaseModel):
    name: str
    aadhaar_number: str = Field(
        pattern=r'^\d{4}\s?\d{4}\s?\d{4}$')
    dob: Optional[str] = None
    gender: Optional[str] = None
    address: Optional[str] = None

class PANCard(BaseModel):
    name: str
    pan_number: str = Field(
        pattern=r'^[A-Z]{5}\d{4}[A-Z]$')
    father_name: Optional[str] = None
    dob: Optional[str] = None

class UtilityBill(BaseModel):
    provider: str
    account_number: str
    billing_period: str
    amount_due: float
    due_date: str
    consumer_name: Optional[str] = None

# Validate sample data
samples = [
    AadhaarCard(name='Rajesh Kumar',
                aadhaar_number='1234 5678 9012',
                dob='1990-05-15', gender='Male',
                address='42 MG Road, Hyderabad'),
    AadhaarCard(name='Priya Sharma',
                aadhaar_number='9876 5432 1098',
                dob='1985-11-20', gender='Female',
                address='15 Banjara Hills, Hyderabad'),
    PANCard(name='Rajesh Kumar',
            pan_number='ABCDE1234F',
            father_name='Suresh Kumar',
            dob='1990-05-15'),
    PANCard(name='Priya Sharma',
            pan_number='FGHIJ5678K',
            father_name='Ramesh Sharma'),
    UtilityBill(provider='TSSPDCL',
                account_number='HYD-2024-78901',
                billing_period='2026-04',
                amount_due=2450.00,
                due_date='2026-05-15',
                consumer_name='Rajesh Kumar'),
    UtilityBill(provider='HMWSSB',
                account_number='WAT-2026-34567',
                billing_period='2026-04',
                amount_due=890.50,
                due_date='2026-05-20',
                consumer_name='Priya Sharma'),
]

print('Indian Document Validation:')
print('=' * 55)
for s in samples:
    cls = s.__class__.__name__
    if cls == 'AadhaarCard':
        print(f'  Aadhaar: {s.name} | {s.aadhaar_number}')
    elif cls == 'PANCard':
        print(f'  PAN:     {s.name} | {s.pan_number}')
    elif cls == 'UtilityBill':
        print(f'  Bill:    {s.provider} | '
              f'Rs {s.amount_due:,.2f}')

print(f'\nAll {len(samples)} documents validated!')

# Test invalid data
import re
try:
    bad = PANCard(name='Test', pan_number='INVALID')
    print('ERROR: should have failed!')
except Exception as e:
    print(f'Invalid PAN rejected: {str(e)[:60]}...')

try:
    bad = AadhaarCard(name='Test', aadhaar_number='123')
    print('ERROR: should have failed!')
except Exception as e:
    print(f'Invalid Aadhaar rejected: {str(e)[:60]}...')


</details>


---
## Exercise 8 (Challenge): Indian Agricultural Advisor

Crop disease database with Hindi. Pure Python.


In [ ]:
# YOUR CODE
# TODO: create diseases list
# TODO: print formatted table


<details><summary>Solution (database, no API)</summary>


In [ ]:
diseases = [
    {'name': 'Late Blight',
     'hindi': '\u0932\u0947\u091f \u092c\u094d\u0932\u093e\u0907\u091f',
     'crop': 'Tomato/Potato',
     'symptoms': 'Dark brown spots, white mold',
     'treatment': 'Apply Mancozeb fungicide',
     'treatment_hindi': '\u092e\u0948\u0902\u0915\u094b\u091c\u0947\u092c \u0915\u0935\u0915\u0928\u093e\u0936\u0940 \u0932\u0917\u093e\u090f\u0902'},
    {'name': 'Powdery Mildew',
     'hindi': '\u092a\u093e\u0909\u0921\u0930\u0940 \u092e\u093f\u0932\u094d\u0921\u094d\u092f\u0942',
     'crop': 'Wheat/Peas',
     'symptoms': 'White powdery coating on leaves',
     'treatment': 'Spray sulfur-based fungicide',
     'treatment_hindi': '\u0938\u0932\u094d\u092b\u0930 \u0906\u0927\u093e\u0930\u093f\u0924 \u0915\u0935\u0915\u0928\u093e\u0936\u0940 \u0915\u093e \u091b\u093f\u0921\u093c\u0915\u093e\u0935 \u0915\u0930\u0947\u0902'},
    {'name': 'Bacterial Leaf Blight',
     'hindi': '\u092c\u0948\u0915\u094d\u091f\u0940\u0930\u093f\u092f\u0932 \u0932\u0940\u092b \u092c\u094d\u0932\u093e\u0907\u091f',
     'crop': 'Rice',
     'symptoms': 'Yellow-white lesions, wilting',
     'treatment': 'Copper-based bactericide',
     'treatment_hindi': '\u0924\u093e\u0902\u092c\u093e \u0906\u0927\u093e\u0930\u093f\u0924 \u091c\u0940\u0935\u093e\u0923\u0941\u0928\u093e\u0936\u0915 \u0932\u0917\u093e\u090f\u0902'},
    {'name': 'Fusarium Wilt',
     'hindi': '\u092b\u094d\u092f\u0942\u091c\u0947\u0930\u093f\u092f\u092e \u0935\u093f\u0932\u094d\u091f',
     'crop': 'Banana/Cotton',
     'symptoms': 'Yellowing, vascular browning',
     'treatment': 'Use resistant varieties, soil solarization',
     'treatment_hindi': '\u092a\u094d\u0930\u0924\u093f\u0930\u094b\u0927\u0940 \u0915\u093f\u0938\u094d\u092e\u0947\u0902 \u0909\u0917\u093e\u090f\u0902'},
    {'name': 'Stem Borer',
     'hindi': '\u0924\u0928\u093e \u091b\u0947\u0926\u0915',
     'crop': 'Rice/Sugarcane',
     'symptoms': 'Dead heart, white ear head',
     'treatment': 'Neem oil spray, pheromone traps',
     'treatment_hindi': '\u0928\u0940\u092e \u0924\u0947\u0932 \u0915\u093e \u091b\u093f\u0921\u093c\u0915\u093e\u0935 \u0915\u0930\u0947\u0902'},
]

print('Crop Disease Database:')
print(f"{'Disease':<24} {'Hindi':<22} "
      f"{'Crop':<16} {'Treatment'}")
print('=' * 95)
for d in diseases:
    print(f"{d['name']:<24} {d['hindi']:<22} "
          f"{d['crop']:<16} {d['treatment'][:30]}...")

print(f'\nTotal diseases: {len(diseases)}')
crops = set()
for d in diseases:
    for c in d['crop'].split('/'):
        crops.add(c.strip())
print(f'Crops covered: {", ".join(sorted(crops))}')

# Hindi output test
print(f'\nHindi Advisory Sample:')
d = diseases[0]
print(f'  \u0930\u094b\u0917: {d["hindi"]}')
print(f'  \u092b\u0938\u0932: {d["crop"]}')
print(f'  \u0909\u092a\u091a\u093e\u0930: {d["treatment_hindi"]}')


</details>


---
## Exercise 1 (Easy): Image Captioning Comparison
Needs Gemini API key or GPU. See practice-lab-6_5.html.


In [ ]:
# Exercise 1: Image Captioning Comparison
# Needs API key / GPU.
pass


---
## Exercise 4 (Medium): Multi-Image Product Comparison
Needs Gemini API key or GPU. See practice-lab-6_5.html.


In [ ]:
# Exercise 4: Multi-Image Product Comparison
# Needs API key / GPU.
pass


---
## Exercise 5 (Medium): VLM Benchmark Evaluation
Needs Gemini API key or GPU. See practice-lab-6_5.html.


In [ ]:
# Exercise 5: VLM Benchmark Evaluation
# Needs API key / GPU.
pass


---
## Exercise 6 (Challenge): Multimodal RAG Pipeline
Needs Gemini API key or GPU. See practice-lab-6_5.html.


In [ ]:
# Exercise 6: Multimodal RAG Pipeline
# Needs API key / GPU.
pass


---
## Exercise 7 (Challenge): Visual QA Chatbot
Needs Gemini API key or GPU. See practice-lab-6_5.html.


In [ ]:
# Exercise 7: Visual QA Chatbot
# Needs API key / GPU.
pass
